In [624]:
import pandas as pd
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

# Reload enriched dataset
df_master = pd.read_csv('nba_master_dataset.csv', parse_dates=['GAME_DATE'])

print(f"Rows:    {len(df_master):,}")
print(f"Columns: {len(df_master.columns)}")
print(f"GAME_DATE dtype: {df_master['GAME_DATE'].dtype}")

Rows:    56,985
Columns: 85
GAME_DATE dtype: datetime64[ns]


In [626]:
# Define features that go into the model
# TARGETS
target_cols = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']

# FEATURES
feature_cols = [
    # 5-game rolling averages
    'PTS_roll5', 'REB_roll5', 'AST_roll5', 'BLK_roll5',
    'STL_roll5', 'FG3M_roll5', 'MIN_roll5', 'TOV_roll5',
    'FGA_roll5', 'FG3A_roll5',

    # 10-game rolling averages
    'PTS_roll10', 'REB_roll10', 'AST_roll10', 'BLK_roll10',
    'STL_roll10', 'FG3M_roll10', 'MIN_roll10', 'TOV_roll10',
    'FGA_roll10', 'FG3A_roll10',

    # Situational context
    'HOME_AWAY',

    # Round 1
    'DAYS_REST',
    'DEF_RATING',
    'PACE',

    # Round 2
    'OPP_PTS_ALLOWED_PG',
    'OPP_REB_ALLOWED_PG',
    'OPP_AST_ALLOWED_PG',
    'OPP_BLK_PG',
    'OPP_STL_PG',
    'OPP_3PM_ALLOWED_PG',

    # Round 3
    'USG_PCT',
    'OPP_PTS_VS_POS',
    'OPP_REB_VS_POS',
    'OPP_AST_VS_POS',
    'OPP_BLK_VS_POS',
    'OPP_STL_VS_POS',
    'OPP_3PM_VS_POS',

    # New rolling features - position normalized
    'USG_PCT_roll5',
    'OPP_PTS_VS_POS_roll5_norm',
    'OPP_REB_VS_POS_roll5_norm',
    'OPP_AST_VS_POS_roll5_norm',
    'OPP_BLK_VS_POS_roll5_norm',
    'OPP_STL_VS_POS_roll5_norm',
    'OPP_3PM_VS_POS_roll5_norm',

    # Playoff flag
    'IS_PLAYOFF',

    # Team usage context
    'RELATIVE_USG',
    'USG_RANK',

    # Volatility features
    'PTS_std_roll10',
    'REB_std_roll10',
    'PTS_cv_roll10',
    'REB_cv_roll10',
]

print(f"Total features: {len(feature_cols)}")
print(f"Target outputs: {len(target_cols)}")

Total features: 51
Target outputs: 6


In [628]:
# Use to convert strings into something PyTorch can understand
# Convert HOME = 1, AWAY = 0
df_master['HOME_AWAY'] = df_master['HOME_AWAY'].map({'HOME': 1, 'AWAY': 0})

# Convert POSITION to numeric — G=0, F=1, C=2
df_master['POSITION'] = df_master['POSITION'].map({'G': 0, 'F': 1, 'C': 2})

# Ensure GAME_DATE is datetime
df_master['GAME_DATE'] = pd.to_datetime(df_master['GAME_DATE'])

# Fix position mixing problem — normalize rolling pos features within each position
# This prevents Centers' naturally higher rebound values from appearing
# as outliers to the scaler when compared to Guards and Forwards
pos_roll_cols = [
    'OPP_PTS_VS_POS_roll5', 'OPP_REB_VS_POS_roll5',
    'OPP_AST_VS_POS_roll5', 'OPP_BLK_VS_POS_roll5',
    'OPP_STL_VS_POS_roll5', 'OPP_3PM_VS_POS_roll5'
]

for col in pos_roll_cols:
    norm_col = f'{col}_norm'
    df_master[norm_col] = df_master.groupby('POSITION')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

# Verify
print(df_master['HOME_AWAY'].value_counts())
print(f"Data type HOME_AWAY: {df_master['HOME_AWAY'].dtype}")
print(f"Data type GAME_DATE: {df_master['GAME_DATE'].dtype}")
print(f"Norm columns added:  {[c for c in df_master.columns if 'norm' in c]}")

HOME_AWAY
1    28590
0    28395
Name: count, dtype: int64
Data type HOME_AWAY: int64
Data type GAME_DATE: datetime64[ns]
Norm columns added:  ['OPP_PTS_VS_POS_roll5_norm', 'OPP_REB_VS_POS_roll5_norm', 'OPP_AST_VS_POS_roll5_norm', 'OPP_BLK_VS_POS_roll5_norm', 'OPP_STL_VS_POS_roll5_norm', 'OPP_3PM_VS_POS_roll5_norm']


In [630]:
# Normalize the features for better model understanding
from sklearn.preprocessing import StandardScaler
import pickle

# Load fresh raw data to refit scaler correctly
# We reload raw because df_master may have been partially transformed
df_raw = pd.read_csv('nba_master_dataset.csv', parse_dates=['GAME_DATE'])
df_raw['HOME_AWAY'] = df_raw['HOME_AWAY'].map({'HOME': 1, 'AWAY': 0})
df_raw['POSITION']  = df_raw['POSITION'].map({'G': 0, 'F': 1, 'C': 2})

# Apply same position normalization to df_raw
for col in pos_roll_cols:
    norm_col = f'{col}_norm'
    df_raw[norm_col] = df_raw.groupby('POSITION')[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

# Drop NaNs before fitting scaler
df_raw = df_raw.dropna(subset=feature_cols).reset_index(drop=True)

# Fit scaler on raw data
scaler = StandardScaler()
scaler.fit(df_raw[feature_cols])

# Verify normalization is working correctly
test       = df_raw[feature_cols].iloc[0:1].values
normalized = scaler.transform(test)

print(f"Scaler refit on {len(feature_cols)} features ✅")
print()
print(f"Raw PTS_roll5:         {test[0][0]:.4f}")
print(f"Normalized PTS_roll5:  {normalized[0][0]:.4f}")
print(f"Scaler mean PTS_roll5: {scaler.mean_[0]:.4f}")
print()
print("After normalization — means should be ~0:")
print(df_raw[feature_cols].describe().round(2)[
    ['OPP_PTS_ALLOWED_PG', 'DEF_RATING', 'PACE', 'USG_PCT']
])

# Save corrected scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print()
print("✅ Corrected scaler saved")

Scaler refit on 51 features ✅

Raw PTS_roll5:         10.2000
Normalized PTS_roll5:  -0.8050
Scaler mean PTS_roll5: 15.7261

After normalization — means should be ~0:
       OPP_PTS_ALLOWED_PG  DEF_RATING      PACE   USG_PCT
count            49449.00    49449.00  49449.00  49449.00
mean               113.30      113.31     99.40      0.21
std                  4.13        2.96      1.82      0.06
min                103.70      106.20     95.64      0.09
25%                110.60      111.50     97.95      0.16
50%                113.60      113.20     99.37      0.20
75%                115.80      115.30    100.62      0.25
max                123.30      119.60    104.67      0.39

✅ Corrected scaler saved


In [632]:
# Drop rows where any feature column has NaN
df_master_clean = df_master.dropna(subset=feature_cols).reset_index(drop=True)

print(f"Rows before dropping NaN: {len(df_master):,}")
print(f"Rows after dropping NaN:  {len(df_master_clean):,}")
print(f"Rows dropped:             {len(df_master) - len(df_master_clean):,}")
print()

# Verify no NaNs remain
remaining_nans = df_master_clean[feature_cols].isna().sum().sum()
print(f"Remaining NaNs: {remaining_nans} ✅" if remaining_nans == 0 else f"Still has NaNs: {remaining_nans} ❌")

Rows before dropping NaN: 56,985
Rows after dropping NaN:  49,449
Rows dropped:             7,536

Remaining NaNs: 0 ✅


In [634]:
# Establishing a train/validation split
# Use 2024-25 season as validation, everything before as training
SPLIT_DATE = '2024-10-01'

df_train = df_master_clean[df_master_clean['GAME_DATE'] < SPLIT_DATE].copy()
df_val   = df_master_clean[df_master_clean['GAME_DATE'] >= SPLIT_DATE].copy()

df_train = df_train.dropna(subset=feature_cols).reset_index(drop=True)
df_val   = df_val.dropna(subset=feature_cols).reset_index(drop=True)

print(f"Training rows:   {len(df_train):,}")
print(f"Validation rows: {len(df_val):,}")
print()
print(f"Norm cols in df_train: {[c for c in df_train.columns if 'norm' in c]}")

Training rows:   36,921
Validation rows: 12,528

Norm cols in df_train: ['OPP_PTS_VS_POS_roll5_norm', 'OPP_REB_VS_POS_roll5_norm', 'OPP_AST_VS_POS_roll5_norm', 'OPP_BLK_VS_POS_roll5_norm', 'OPP_STL_VS_POS_roll5_norm', 'OPP_3PM_VS_POS_roll5_norm']


In [636]:
# Convert to PyTorch tensors — apply scaler transform first
import torch

# Normalize features using the fitted scaler
X_train_raw = df_train[feature_cols].values.astype(np.float32)
X_val_raw   = df_val[feature_cols].values.astype(np.float32)

X_train_norm = scaler.transform(X_train_raw)
X_val_norm   = scaler.transform(X_val_raw)

# Convert to tensors
X_train = torch.tensor(X_train_norm, dtype=torch.float32)
y_train = torch.tensor(df_train[target_cols].values, dtype=torch.float32)

X_val   = torch.tensor(X_val_norm, dtype=torch.float32)
y_val   = torch.tensor(df_val[target_cols].values, dtype=torch.float32)

# Verify shapes and scale
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print()
print(f"X_train range: min={X_train.min():.4f} max={X_train.max():.4f} mean={X_train.mean():.4f}")
print(f"NaNs in X_train: {torch.isnan(X_train).sum().item()}")
print(f"NaNs in y_train: {torch.isnan(y_train).sum().item()}")

X_train shape: torch.Size([36921, 51])
y_train shape: torch.Size([36921, 6])

X_train range: min=-3.9050 max=14.7205 mean=-0.0037
NaNs in X_train: 0
NaNs in y_train: 0


In [638]:
# Define model
import torch.nn as nn
import numpy as np
import os

# Set random seed for reproducibility
torch.manual_seed(123)
np.random.seed(123)

class PlayerPropModel(nn.Module):
    def __init__(self, input_dim, target_stats):
        super().__init__()

        # SHARED TRUNK
        # Learns universal basketball performance patterns
        # Every stat head benefits from what this learns
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.4),
        )

        # STAT-SPECIFIC HEADS
        # One head per target stat
        # Each outputs (mu, log_sigma) — the distribution parameters
        # We predict log_sigma instead of sigma directly
        # because log_sigma can be any number, but sigma must be positive
        self.heads = nn.ModuleDict({
            stat: nn.Sequential(
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 2)    # 2 outputs: mu and log_sigma
            )
            for stat in target_stats
        })

    def forward(self, x):
        # Pass input through shared trunk
        shared = self.trunk(x)

        # Pass trunk output through each stat head
        outputs = {}
        for stat, head in self.heads.items():
            raw       = head(shared)
            mu        = raw[:, 0]
            log_sigma = raw[:, 1]

            # Clamp log_sigma to prevent extreme values
            # exp(-3) ≈ 0.05, exp(3) ≈ 20 — reasonable range for sports stats
            log_sigma = torch.clamp(log_sigma, min=-3, max=3)

            sigma = torch.exp(log_sigma) + 1e-6
            outputs[stat] = (mu, sigma)
        return outputs


# Delete old weights file
if os.path.exists('best_model.pth'):
    os.remove('best_model.pth')
    print("🗑️  Removed old weights")

# Initialize fresh model
model = PlayerPropModel(
    input_dim=len(feature_cols),
    target_stats=target_cols
)

# Apply Xavier initialization to all linear layers
# This gives much more stable starting weights than random
# Prevents the exploding/vanishing gradient problem
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

model.apply(init_weights)
model = model.to(device)

print("✅ Fresh model initialized with Xavier weights and clamped sigma")
print()
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")
print(f"Input features: {len(feature_cols)}")

🗑️  Removed old weights
✅ Fresh model initialized with Xavier weights and clamped sigma

Total trainable parameters: 28,172
Input features: 51


In [640]:
# Define loss function
def nll_loss(mu, sigma, target):
    """
    Negative Log-Likelihood loss for a Normal distribution.
    Instead of penalizing just how wrong the mean prediction is,
    this penalizes the model for being:
      1. Wrong about the mean (mu far from target)
      2. Overconfident (sigma too small when wrong)
      3. Underconfident (sigma too large when right)
    This forces the model to learn well-calibrated uncertainty.
    """
    distribution = torch.distributions.Normal(mu, sigma)
    loss = -distribution.log_prob(target)
    return loss.mean()

# Test it works on dummy data
dummy_mu     = torch.tensor([25.0, 10.0, 5.0])
dummy_sigma  = torch.tensor([3.0,  2.0,  1.0])
dummy_target = torch.tensor([27.0, 9.0,  6.0])

test_loss = nll_loss(dummy_mu, dummy_sigma, dummy_target)
print(f"Test loss: {test_loss.item():.4f}")
print("Loss function working ✅")

Test loss: 1.7986
Loss function working ✅


In [642]:
from torch.utils.data import TensorDataset, DataLoader

# Hyperparameters
EPOCHS        = 30
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3

# Package data into DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset   = TensorDataset(X_val, y_val)
val_loader    = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Optimizer — with L2 regularization
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.7,
    patience=15,
)

# Move model to GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f"Training on: {device}")

# Training loop
train_losses  = []
val_losses    = []
best_val_loss = float('inf')

for epoch in range(EPOCHS):

    # Training phase
    model.train()
    epoch_train_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = sum(
            nll_loss(
                outputs[stat][0],
                outputs[stat][1],
                y_batch[:, i]
            )
            for i, stat in enumerate(target_cols)
        ) / len(target_cols)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_train_loss += loss.item()

    avg_train_loss = epoch_train_loss / len(train_loader)

    # Validation phase
    model.eval()
    epoch_val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = sum(
                nll_loss(
                    outputs[stat][0],
                    outputs[stat][1],
                    y_batch[:, i]
                )
                for i, stat in enumerate(target_cols)
            ) / len(target_cols)

            epoch_val_loss += loss.item()

    avg_val_loss = epoch_val_loss / len(val_loader)

    # Step scheduler
    scheduler.step(avg_val_loss)

    # Track losses
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')

    # Print every 5 epochs
    if (epoch + 1) % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"LR: {current_lr:.6f} | "
              f"{'✅ Best' if avg_val_loss == best_val_loss else ''}")

print()
print(f"Training complete — best val loss: {best_val_loss:.4f}")
print(f"Best model saved to: best_model.pth")

Training on: cpu
Epoch   5/30 | Train Loss: 2.0889 | Val Loss: 2.0215 | LR: 0.001000 | ✅ Best
Epoch  10/30 | Train Loss: 2.0330 | Val Loss: 1.9992 | LR: 0.001000 | ✅ Best
Epoch  15/30 | Train Loss: 2.0134 | Val Loss: 1.9924 | LR: 0.001000 | 
Epoch  20/30 | Train Loss: 2.0090 | Val Loss: 1.9865 | LR: 0.001000 | 
Epoch  25/30 | Train Loss: 1.9995 | Val Loss: 1.9821 | LR: 0.001000 | 
Epoch  30/30 | Train Loss: 1.9968 | Val Loss: 1.9848 | LR: 0.001000 | 

Training complete — best val loss: 1.9795
Best model saved to: best_model.pth


In [644]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Evaluation check using Wemby
# Reload best model weights
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()
print("✅ Best model weights reloaded")
print()

wemby_val = df_master[
    (df_master['PLAYER_NAME'] == 'Victor Wembanyama') &
    (df_master['GAME_DATE'] >= '2024-10-01')
].dropna(subset=feature_cols).copy()

print(f"Wemby games in validation set: {len(wemby_val)}")
print()

wemby_features_raw        = wemby_val[feature_cols].values.astype(np.float32)
wemby_features_normalized = scaler.transform(wemby_features_raw)
wemby_features            = torch.tensor(
    wemby_features_normalized, dtype=torch.float32
).to(device)

with torch.no_grad():
    wemby_preds = model(wemby_features)

print("Wembanyama — Predicted vs Actual Points:")
print(f"{'Game':<12} {'Actual':>8} {'Pred μ':>8} {'Pred σ':>8} {'Error':>8}")
print("-" * 45)

for i in range(min(10, len(wemby_val))):
    actual = wemby_val['PTS'].iloc[i]
    mu     = wemby_preds['PTS'][0][i].item()
    sigma  = wemby_preds['PTS'][1][i].item()
    error  = abs(actual - mu)
    date   = str(wemby_val['GAME_DATE'].iloc[i].date())
    print(f"{date:<12} {actual:>8.1f} {mu:>8.1f} {sigma:>8.1f} {error:>8.1f}")

✅ Best model weights reloaded

Wemby games in validation set: 36

Wembanyama — Predicted vs Actual Points:
Game           Actual   Pred μ   Pred σ    Error
---------------------------------------------
2024-11-11       34.0     21.6      7.3     12.4
2024-11-13       50.0     23.6      7.6     26.4
2024-11-15       28.0     21.5      7.4      6.5
2024-11-23       25.0     22.1      7.5      2.9
2024-11-26       34.0     24.1      7.6      9.9
2024-11-27       20.0     22.8      7.6      2.8
2024-12-01       34.0     23.6      7.7     10.4
2024-12-03       15.0     25.1      8.2     10.1
2024-12-08       25.0     24.6      8.1      0.4
2024-12-13       28.0     23.9      8.0      4.1
